# p109 late-training reorganization (~27k)

`p109_seed485_dseed598` grokked early, sat stable ~5k–26k, then spiked hard at ~27k and re-settled by ~30k. Three instruments (parameter PCA, activation-DMD eigenvalues, DMD residuals) fire in the same window, but the committed frequencies (4/14/27) survive. This notebook works the event in seven passes:

1. **Event localization** — which neurons explode, and how concentrated is it?
2. **Grouping reconciliation** — the clustering artifact vs. the distribution view partition neurons differently. Pin down the rule before counting anything.
3. **Frequency switching** — did neurons change frequency? Gate on confidence to separate real reassignment from clustering jitter.
4. **Freq-8 transient + attention lead** — the sub-dominant swell that complicates 'none born', and the propagation order.
5. **Procrustes: gauge vs functional** — is the upstream (embedding) move a rotation the MLP mostly ignores, leaving the ~11 exploders as the functional residue?
6. **Gauge-blind geometry** — rotation-invariant centroid distances localize the *functional* change: does the MLP's 2D→1D operand collapse survive while attention reorganizes?
7. **Single-neuron spectral fingerprint** — the blow-up at neuron resolution: a transient collapse to the bare frequency envelope at 27400, then an activation-magnitude detonation with structure preserved.

First pass lives in `apps/research/sketches/p109_event_neuron_displacement.py`; this notebook lifts and extends it. All data access goes through the miscope API (`variant.artifacts`), not file paths.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

# Locate the repo root (holds data/) and chdir there so relative data paths resolve
# from any launch directory; then make the first-pass sketch importable.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "modulo_addition_1layer").exists())
os.chdir(root)
sys.path.insert(0, str(root / "apps" / "research" / "sketches"))
import p109_event_neuron_displacement as fp  # load_w_in_trajectory, freq_assignments, windows
from miscope.families.discovery import load_family_from_dir
from miscope.views.universal import _adapt_activation_freq_legacy

fam = load_family_from_dir("data/modulo_addition_1layer", "data")
variant = fam.get_variant(prime=109, seed=485, data_seed=598)
variant

## 1. Event localization — which neurons explode?

Rank neurons by peak `W_in`-column excursion during the event, in units of each neuron's own normal plateau drift. The handful of outliers carry the globally-visible weight-space departure.

In [ ]:
epochs, W = fp.load_w_in_trajectory(variant)          # (E, N, D)
assign = fp.freq_assignments(variant, fp.GROUP_EPOCH)  # row-index per neuron (see section 2)

plateau = fp._window_mask(epochs, *fp.PLATEAU)
event = fp._window_mask(epochs, *fp.EVENT)
post_idx = int(np.argmin(np.abs(epochs - fp.POST)))

ref = W[plateau].mean(axis=0)
normal_step = np.linalg.norm(np.diff(W[plateau], axis=0), axis=2).mean(axis=0) + 1e-9
peak_disp = np.linalg.norm(W[event] - ref[None], axis=2).max(axis=0)
net_disp = np.linalg.norm(W[post_idx] - ref, axis=1)
score = peak_disp / normal_step

order = np.argsort(score)[::-1]
print("top exploders (neuron, freq-row, score×, peak, net):")
for j in order[:15]:
    print(f"  {j:>3}  row={assign[j]:>2}  {score[j]:>8.0f}×  peak={peak_disp[j]:.2f}  net={net_disp[j]:.2f}")

In [ ]:
# Per-epoch excursion traces for the top movers — do they leave and return, or relocate?
fig = go.Figure()
for j in order[:6]:
    disp = np.linalg.norm(W - ref[None], axis=2)[:, j]
    fig.add_trace(go.Scatter(x=epochs, y=disp, mode="lines", name=f"n{j} (row {assign[j]})"))
fig.update_layout(title="Top-6 exploder W_in excursion from plateau reference",
                  xaxis_title="epoch", yaxis_title="||W_in[:,j] - ref||",
                  xaxis_range=[24000, 30000], height=420)
fig.show()

## 2. Grouping reconciliation

Two surfaces partition neurons by frequency and they disagree:

- `neuron_grouping.assignments` clusters all 512 neurons — sizes ~274/95/143.
- The `neuron_freq_distribution` view (committed/dominant rule) — sizes 183/134/195 in the plots.

**Resolved indexing:** `mlp_out_frequencies` maps row index `k → frequency k+1`. So the grouping's `3/13/26` are *row indices* = actual frequencies **4/14/27**, matching the plot labels. The count gap is a genuine membership-rule difference (cluster-assignment vs. dominant-frequency), not an indexing bug. Decide which rule is authoritative for switch-counting below.

In [ ]:
def norm_matrix(epoch):
    art = variant.artifacts.load_epoch("activation_basis_projection", epoch)
    return _adapt_activation_freq_legacy(art, "mlp_out", "norm_matrix")["norm_matrix"]  # (F, N)

freqs = variant.artifacts.load_epoch("activation_basis_projection", fp.GROUP_EPOCH)["mlp_out_frequencies"]
print("row->freq map (first 16):", list(zip(range(16), freqs[:16].tolist())))

# Cluster sizes (row indices) vs dominant-frequency membership at the same epoch.
u, c = np.unique(assign, return_counts=True)
print("\ncluster sizes (row -> count):", {int(k): int(v) for k, v in zip(u, c)})
nm = norm_matrix(fp.GROUP_EPOCH)
dom = np.argmax(nm, axis=0)
u2, c2 = np.unique(dom, return_counts=True)
top = sorted(zip(u2.tolist(), c2.tolist()), key=lambda x: -x[1])[:5]
print("dominant-freq sizes (row -> count, top 5):", {int(freqs[k] - 1): v for k, v in top}, "<- as rows")

## 3. Frequency switching (confidence-gated)

The switch guess, tested. Group sizes do shift directionally (freq-14 grows, freq-27 shrinks), but the switchers are the low-confidence neurons — the committed core holds.

In [ ]:
fp.frequency_switch_report(variant)

## 4. Freq-8 transient + attention lead

Frequency 8 (row index 7) swells ~2× across 28100–28700 then recedes — but never becomes any neuron's dominant frequency. A threshold-sensitive 'none born' caveat. Separately, attention-out's DMD residual climbs before MLP-out's spike; freq-8 also shows up as a dominant *pair* in the attention spectra.

In [ ]:
i8 = int(np.where(freqs == 8)[0][0])  # actual frequency 8 -> row 7
win = [e for e in epochs if 26500 <= e <= 30000]
power8, dom8 = [], []
for e in win:
    m = norm_matrix(int(e))
    power8.append(m[i8].sum())
    d = np.argmax(m, axis=0)
    df = m[d, np.arange(m.shape[1])]
    dom8.append(int(((d == i8) & (df >= 0.10)).sum()))

fig = go.Figure()
fig.add_trace(go.Scatter(x=win, y=power8, mode="lines+markers", name="freq-8 total power"))
fig.add_trace(go.Scatter(x=win, y=dom8, mode="lines+markers", name="neurons dominant@10%", yaxis="y2"))
fig.update_layout(title="Freq-8: transient power swell, zero dominance",
                  xaxis_title="epoch", yaxis_title="total power",
                  yaxis2=dict(title="dominant-neuron count", overlaying="y", side="right"),
                  height=420)
fig.show()

## 5. Procrustes — separating gauge from functional movement

Hypothesis (from the architecture): attention+embeddings lead the event because they're one coupled subsystem with the most reparameterization freedom, so under weight decay they drift first along a **gauge manifold** (a rotation of the `d_model` basis) that mostly preserves the signal the MLP reads. The MLP stays put to the extent the upstream move was gauge; the ~11 exploders are the small **functional residue** that wasn't.

**Method.** Orthogonal Procrustes recovers the best `d_model` rotation `R` from the embedding move (`W_E_pre @ R ≈ W_E_post`). If the move is largely rotational, `gauge-explained` is high. The gauge-consistent MLP co-rotation is then `W_in_post ≈ Rᵀ @ W_in_pre` (keeps `hidden = resid @ W_in` invariant); the per-neuron residual after co-rotation isolates the functional component — checked against the section-1 exploders.

**Caveat.** A *full* network gauge symmetry needs `W_Q/W_K/W_V/W_O/W_out` to co-rotate too; this tests the embedding↔MLP-input pair, the cleanest slice. The negative MLP gauge fraction below is exactly this caveat biting — attention mediates between embedding and MLP (picked up in §6).

In [ ]:
from scipy.linalg import orthogonal_procrustes


def gauge_split(A, B):
    """Best orthogonal R with A@R ~ B; how much of the A->B move is rotational."""
    R, _ = orthogonal_procrustes(A, B)
    raw = np.linalg.norm(B - A)
    resid = np.linalg.norm(B - A @ R)
    gauge_frac = 1 - (resid / raw) ** 2 if raw > 0 else float("nan")
    return R, raw, resid, gauge_frac


PRE, POST = 26000, int(epochs[post_idx])
we = lambda e: variant.artifacts.load_epoch("parameter_snapshot", e)["W_E"].astype(np.float64)

R, raw, resid, gf = gauge_split(we(PRE), we(POST))       # embedding move across the event
R0, raw0, resid0, gf0 = gauge_split(we(24000), we(26000))  # plateau null for calibration

print("W_E embedding move — d_model rotation via orthogonal Procrustes (W_E_pre @ R ~ W_E_post)")
print(f"  event {PRE}->{POST}: ||ΔW_E||={raw:7.3f}  aligned resid={resid:7.3f}  gauge-explained={gf:6.1%}")
print(f"  null  24000->26000 : ||ΔW_E||={raw0:7.3f}  aligned resid={resid0:7.3f}  gauge-explained={gf0:6.1%}")
print(f"  rotation size ||R-I||_F (event) = {np.linalg.norm(R - np.eye(R.shape[0])):.3f}")

In [ ]:
# If the embedding basis rotates by R (resid' = resid @ R), the gauge-consistent MLP
# co-rotation is W_in' = Rᵀ @ W_in (keeps hidden = resid @ W_in invariant). The
# residual after co-rotation is the part of the MLP move NOT explained by the
# embedding rotation — the functional component. Test: is it carried by the
# section-1 exploders?
pre_idx = int(np.argmin(np.abs(epochs - PRE)))
W_in_pre, W_in_post = W[pre_idx].T, W[post_idx].T          # (d_model, n_neurons)
pred = R.T @ W_in_pre                                       # gauge prediction
raw_col = np.linalg.norm(W_in_post - W_in_pre, axis=0)      # per-neuron raw move
resid_col = np.linalg.norm(W_in_post - pred, axis=0)        # per-neuron co-rotation residual
mlp_gauge_frac = 1 - (np.linalg.norm(W_in_post - pred) / np.linalg.norm(W_in_post - W_in_pre)) ** 2
print(f"MLP move gauge-explained by the embedding rotation: {mlp_gauge_frac:.1%}")

exploders = list(order[:11])
resid_rank = np.argsort(resid_col)[::-1].tolist()
overlap = len(set(resid_rank[:11]) & set(exploders))
print(f"top-11 by co-rotation residual: {resid_rank[:11]}")
print(f"section-1 exploders          : {exploders}")
print(f"overlap: {overlap}/11  (high overlap => exploders ARE the functional residue)")

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(resid_col)), y=resid_col, mode="markers",
                         name="all neurons", marker=dict(size=4, color="lightgray")))
fig.add_trace(go.Scatter(x=exploders, y=resid_col[exploders], mode="markers",
                         name="section-1 exploders", marker=dict(size=10, color="crimson")))
fig.update_layout(title="Per-neuron MLP residual after embedding-rotation co-rotation",
                  xaxis_title="neuron", yaxis_title="||W_in_post - Rᵀ·W_in_pre||", height=420)
fig.show()

## 6. Gauge-blind geometry — where the functional change actually lands

Centroid distances and Fisher are **rotation-invariant**, so they cannot see the §5 embedding rotation (86% of the upstream move) — they isolate the functional change. Two readouts across 25000→28000:

- **Per-site 2D-residual fraction** — how much of each site's centroid-distance matrix is *not* a function of `(i−j)`. At 26300 this is ~22% for `attn_out` (attention still carries the operands `a`, `b` as a 2D pair) vs ~1% for `mlp_out`/`resid_post` (the MLP has collapsed it to the 1D result `a+b`). The MLP **is** that 2D→1D collapse. If `mlp_out` stays flat near ~1% through the event, the collapse — the MLP's core function — survived (consistent with frequencies maintained + only 11 neurons moving). If `attn_out`'s fraction moves, that's the upstream functional disturbance.
- **`resid_post` distance-matrix drift** — frame-to-frame change of the distance matrix (rotation cancels), timing the functional reorganization against the 27400→27600 disorganize/heal window seen in the centroid-PCA ring.

In [ ]:
# Centroid distances + Fisher are rotation-invariant: blind to the §5 embedding
# rotation (86% of the upstream move), sensitive only to the functional change.
# Two gauge-blind readouts across the event window.
SITES = ["attn_out", "mlp_out", "resid_post"]
geo_eps = [int(e) for e in epochs if 25000 <= e <= 28000]


def centroid_dist(site, epoch):
    C = variant.artifacts.load_epoch("repr_geometry", int(epoch))[f"{site}_centroids"].astype(np.float64)
    return np.linalg.norm(C[:, None, :] - C[None, :, :], axis=2)


def two_d_residual(D):
    """Fraction of off-diagonal distance variance NOT a function of (i-j) mod n (the 2D part)."""
    n = D.shape[0]
    ij = (np.arange(n)[:, None] - np.arange(n)[None, :]) % n
    g = np.array([D[ij == k].mean() for k in range(n)])
    off = ~np.eye(n, dtype=bool)
    return float(np.mean((D[off] - g[ij][off]) ** 2) / np.var(D[off]))


twod = {s: [two_d_residual(centroid_dist(s, e)) for e in geo_eps] for s in SITES}
# resid_post distance-matrix drift (rotation cancels in distances → functional only).
Dr = [centroid_dist("resid_post", e) for e in geo_eps]
drift = [0.0] + [float(np.linalg.norm(Dr[i] - Dr[i - 1]) / np.linalg.norm(Dr[i - 1]))
                 for i in range(1, len(Dr))]

fig = go.Figure()
for s in SITES:
    fig.add_trace(go.Scatter(x=geo_eps, y=twod[s], mode="lines+markers", name=f"{s} 2D-residual"))
fig.add_trace(go.Scatter(x=geo_eps, y=drift, mode="lines+markers", name="resid_post drift",
                         yaxis="y2", line=dict(dash="dot", color="black")))
fig.update_layout(title="Gauge-blind geometry across the event (2D-residual fraction + resid_post drift)",
                  xaxis_title="epoch", yaxis_title="2D-residual fraction",
                  yaxis2=dict(title="resid_post distance drift", overlaying="y", side="right"),
                  height=440)
fig.show()

for s in SITES:
    print(f"{s:>12}: 2D-residual {twod[s][0]:5.1%} (start) -> {max(twod[s]):5.1%} (peak) -> {twod[s][-1]:5.1%} (end)")
print(f"resid_post peak drift at epoch {geo_eps[int(np.argmax(drift))]}")

## 7. Single-neuron spectral fingerprint — the blow-up at neuron resolution

Each neuron's `(a,b)` activation map has a 2D Fourier signature. Four scannable numbers per neuron: **power** (activation scale), **dominant frequency**, **result-alignment** (power on `f_a=f_b`, the `a+b` result, vs the `f_a+f_b≡0` lattice / `a−b` line), and **on-lines** (fraction of total power on those two principal lines = spectral cleanliness).

The two §1 exploders read as opposite frequencies — **n327 → freq 4** (coarse 4×4 block lattice), **n400 → freq 27** (fine dots) — both ~24–28th percentile in result-alignment (more lattice than the 0.76 population median). Tracked through the event, the fingerprint *is* the blow-up at neuron resolution: at **27400–27500** n327 momentarily collapses to its bare freq-4 envelope (result-align 0.83→0.60, the fine structure drops out), **recovers by 27600**, then its activation power detonates ~**800×** (std ~0.16→~4) while the frequency/result structure returns unchanged. So the §1 *weight* explosion surfaces here as an *activation-magnitude* explosion with structure preserved — implying `W_out` down-scales the blown-up feature (open thread). The frame sequence at 27300→27700 shows exactly this: dense lattice → bare envelope (trough) → dense → dense+large.

In [ ]:
import scipy.stats as ss

# Per-neuron 2D-spectrum fingerprint over the (a,b) activation map: power (scale),
# result-alignment (power on f_a=f_b, the a+b result, vs the f_a+f_b≡0 lattice line),
# and on-lines (how much total power sits on those two principal lines = cleanliness).
PG = 109
FA, FB = np.meshgrid(np.arange(PG), np.arange(PG), indexing="ij")
DC = (FA == 0) & (FB == 0)
SUM_LINE = (FA == FB) & ~DC          # a+b (result) structure
DIFF_LINE = ((FA + FB) % PG == 0) & ~DC  # a-b (lattice) structure


def neuron_fingerprint(epoch):
    A = variant.artifacts.load_epoch("neuron_activations", int(epoch))["activations"].astype(np.float64)
    A = A - A.mean(axis=(1, 2), keepdims=True)
    Pw = np.abs(np.fft.fft2(A, axes=(1, 2))) ** 2          # (512, p, p)
    tot = Pw.sum((1, 2)) - Pw[:, 0, 0]
    psum, pdiff = Pw[:, SUM_LINE].sum(1), Pw[:, DIFF_LINE].sum(1)
    return tot, psum / (psum + pdiff + 1e-12), (psum + pdiff) / (tot + 1e-12)


fp_eps = [int(e) for e in epochs if 26000 <= e <= 28000]
traj = {n: {"power": [], "align": []} for n in (327, 400)}
for e in fp_eps:
    tot, align, _ = neuron_fingerprint(e)
    for n in (327, 400):
        traj[n]["power"].append(tot[n])
        traj[n]["align"].append(align[n])

fig = go.Figure()
for n in (327, 400):
    fig.add_trace(go.Scatter(x=fp_eps, y=traj[n]["power"], mode="lines+markers", name=f"n{n} power"))
    fig.add_trace(go.Scatter(x=fp_eps, y=traj[n]["align"], mode="lines+markers",
                             name=f"n{n} result-align", yaxis="y2", line=dict(dash="dot")))
fig.update_layout(title="Exploder fingerprint through the event: activation power (log) + result-alignment",
                  xaxis_title="epoch", yaxis_title="activation power", yaxis_type="log",
                  yaxis2=dict(title="result-alignment", overlaying="y", side="right", range=[0, 1]),
                  height=440)
fig.show()

# Population scan at a settled post-event epoch: where do the exploders rank?
tot, align, lines = neuron_fingerprint(29999)
for n in (327, 400):
    print(f"n{n} @29999: power pct={ss.percentileofscore(tot, tot[n]):.0f}  "
          f"result-align={align[n]:.2f} (pct {ss.percentileofscore(align, align[n]):.0f})  on-lines={lines[n]:.0%}")
trough_e = fp_eps[int(np.argmin(traj[327]["align"]))]
print(f"n327 result-align trough: {min(traj[327]['align']):.2f} at epoch {trough_e}; "
      f"power {traj[327]['power'][0]:.2e} -> {traj[327]['power'][-1]:.2e}")

### Open threads

- Do the ~11 exploders leave-and-return or genuinely relocate? (peak vs net already disagree)
- Settle the authoritative grouping rule (section 2) before any published switch count.
- Are the exploded neurons the ones attention was routing to when its residual started climbing (~23–24k)?
- Why does loosened capacity re-commit preferentially to freq-14?
- **Fold the attention path into the Procrustes test (§5).** Recover `R` from `W_E`, verify `W_Q/W_K/W_V` co-rotate by `Rᵀ`, then measure the MLP residual against the *attention-output* frame, not the embedding frame — the negative direct-co-rotation gauge fraction says attention mediates.
- **§6 finding to chase:** `attn_out`'s 2D-residual *fell* ~20%→11% across the event while `mlp_out` stayed flat at ~1–2% — attention shed operand structure while the MLP's collapse held. Is the drop permanent or does it recover after 28000?
- **§7 follow-ups:** (a) the ~800× activation-magnitude detonation with structure preserved implies `W_out` down-scales these features — verify the `W_out` rows of the exploders shrink to compensate. (b) The 4-number fingerprint (power, dom-freq, result-align, on-lines) is a candidate `neuron_activation_spectrum` analyzer (declared per-neuron outputs) for scanning the whole corpus.